# Step 6 · wb1e5: White-box attack on **multilingual-E5 + para** (MCP-Guard-style classifier). Can run in parallel with wb1.
Runs `scripts/attack_whitebox.py`: trains the target detector (~15 min), attacks every control-verified working attack
with gradient-guided word swaps, then re-checks every evader on the Qwen3-8B agent with and without the tool.

**Before you run**
1. Push commit `058d739` (or later) to GitHub.
2. Settings: Accelerator **GPU T4 x2**, Internet **On**, Persistence: none needed.
3. Add-ons → Secrets: tick **Hugging Face** and **mcp-git** for *this* notebook (a copied notebook loses them).
4. **Save Version → Save & Run All** (runs in the background, up to 12 h).

**After it finishes:** Output tab → Download All (zip) → put the zip in the repo folder.
Headline number: `n_genuine_evasions` in `summary.json` (printed in the last cell).

**If the session dies:** attach this notebook's output as an input Dataset, set `RESUME_FROM` below to its path, re-run.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/attack_whitebox.py"
ARGS = "--tag wb1e5 --set detector.backbone=intfloat/multilingual-e5-base"
GPU_DEPS = True          # bitsandbytes for the 4-bit Qwen3-8B agent (verify phase)
MIN_COMMIT_FILE = "scripts/attack_whitebox.py"   # sanity check that the pushed code is new enough

# Resume: path to the previous run's output Dataset (the folder that contains outputs/), else None
RESUME_FROM = None       # e.g. "/kaggle/input/wb1-output"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push commit 058d739 first"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2: detector on cuda:0, agent on cuda:1"

run_args = ARGS
if RESUME_FROM:
    import glob, shutil
    shutil.copytree(os.path.join(RESUME_FROM, "outputs"), os.path.join(WORK, "outputs"), dirs_exist_ok=True)
    prev = sorted(glob.glob(os.path.join(WORK, "outputs", "attack_whitebox", "*")))
    assert prev, "no previous attack_whitebox run found in RESUME_FROM"
    run_args += f" --resume {os.path.basename(prev[-1])}"
    print("resuming", prev[-1])
print("python", SCRIPT, run_args)


In [ ]:
!python {SCRIPT} {run_args}


In [ ]:
import glob, json
run = sorted(glob.glob("/kaggle/working/outputs/attack_whitebox/*"))[-1]
print(run)
print(json.dumps(json.load(open(f"{run}/summary.json")), indent=2))
# Example evading rewrites (if any)
for l in open(f"{run}/whitebox.jsonl"):
    r = json.loads(l)
    if r["evaded"]:
        print(f"\n[{r['round']}] margin {r['margin_orig']:+.2f} -> {r['margin_adv']:+.2f}, {r['n_changed']} words: {r['changes']}")
